# Phase 10: TabularGRPO Fraud Research

**Phase:** 19

Combine a tabular Transformer, Mixture of Experts, and GRPO for fraud decision policy research.

This notebook preserves the project contract established in Phase 1 and Phase 2:

- Permanent code in GitHub
- Permanent dataset and artifacts in Google Drive
- ULB `creditcard.csv`
- Chronological 60/20/20 partitions
- Training-only preprocessing
- Validation-only model and threshold selection
- One final test evaluation
- Reproducible artifacts and acceptance checks

> Research POC only. Do not use the outputs for live transaction decisions.


## Storage and execution model

- GitHub repository: `deepak-malviyaa/enterprise-adaptive-learning-lab`
- Dataset: `MyDrive/enterprise-adaptive-learning-lab-data/raw/creditcard.csv`
- Runs: `MyDrive/enterprise-adaptive-learning-lab-data/outputs/runs/`
- Colab `/content`: temporary runtime only


In [ ]:
# Common setup: Drive, GitHub, project installation, and paths
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime, timezone
import os, sys, json, time, random, shutil, subprocess, hashlib, platform

SEED=42
random.seed(SEED)
REPO_URL='https://github.com/deepak-malviyaa/enterprise-adaptive-learning-lab.git'
PROJECT_ROOT=Path('/content/enterprise-adaptive-learning-lab')
DRIVE_ROOT=Path('/content/drive/MyDrive/enterprise-adaptive-learning-lab-data')
DATA_PATH=DRIVE_ROOT/'raw'/'creditcard.csv'
RUNS_ROOT=DRIVE_ROOT/'outputs'/'runs'
RUNS_ROOT.mkdir(parents=True,exist_ok=True)

if (PROJECT_ROOT/'.git').exists():
    result=subprocess.run(['git','-C',str(PROJECT_ROOT),'pull','--ff-only'],text=True,capture_output=True)
elif PROJECT_ROOT.exists():
    shutil.rmtree(PROJECT_ROOT)
    result=subprocess.run(['git','clone',REPO_URL,str(PROJECT_ROOT)],text=True,capture_output=True)
else:
    result=subprocess.run(['git','clone',REPO_URL,str(PROJECT_ROOT)],text=True,capture_output=True)
print(result.stdout); print(result.stderr)
if result.returncode!=0: raise RuntimeError('Git operation failed.')

pyproject=PROJECT_ROOT/'pyproject.toml'
text=pyproject.read_text(encoding='utf-8').replace('>=3.10,<3.13','>=3.10,<3.14')
pyproject.write_text(text,encoding='utf-8')
os.chdir(PROJECT_ROOT)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.'],check=True)
if not DATA_PATH.exists(): raise FileNotFoundError(f'Missing permanent dataset: {DATA_PATH}')
print('Project and dataset ready.')


In [ ]:
# Common data contract, checksum, chronological split, and preprocessing
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score,roc_auc_score,precision_score,recall_score,f1_score,brier_score_loss,confusion_matrix

np.random.seed(SEED)
def sha256_file(path):
    d=hashlib.sha256()
    with Path(path).open('rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''): d.update(b)
    return d.hexdigest()

DATASET_SHA256=sha256_file(DATA_PATH)
df=pd.read_csv(DATA_PATH)
EXPECTED=['Time']+[f'V{i}' for i in range(1,29)]+['Amount','Class']
missing=sorted(set(EXPECTED)-set(df.columns))
if missing or set(df.Class.unique())!={0,1}: raise ValueError(f'Invalid dataset. Missing={missing}')

df=df.sort_values('Time',kind='mergesort').reset_index(drop=True)
df['row_id']=np.arange(len(df))
a=int(len(df)*.60); b=int(len(df)*.80)
train_df=df.iloc[:a].copy(); val_df=df.iloc[a:b].copy(); test_df=df.iloc[b:].copy()
FEATURES=['Time']+[f'V{i}' for i in range(1,29)]+['Amount']
X_train=train_df[FEATURES].copy(); y_train=train_df.Class.to_numpy()
X_val=val_df[FEATURES].copy(); y_val=val_df.Class.to_numpy()
X_test=test_df[FEATURES].copy(); y_test=test_df.Class.to_numpy()
scaler=StandardScaler().fit(X_train[['Time','Amount']])
for frame in [X_train,X_val,X_test]: frame.loc[:,['Time','Amount']]=scaler.transform(frame[['Time','Amount']])
X_train=X_train.to_numpy(np.float32); X_val=X_val.to_numpy(np.float32); X_test=X_test.to_numpy(np.float32)
print('Checksum:',DATASET_SHA256); print(X_train.shape,X_val.shape,X_test.shape)

def metrics(y,p,threshold=.5):
    pred=(p>=threshold).astype(int); tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    return {'pr_auc':float(average_precision_score(y,p)),'roc_auc':float(roc_auc_score(y,p)),'precision':float(precision_score(y,pred,zero_division=0)),'recall':float(recall_score(y,pred,zero_division=0)),'f1':float(f1_score(y,pred,zero_division=0)),'brier':float(brier_score_loss(y,p)),'threshold':float(threshold),'tn':int(tn),'fp':int(fp),'fn':int(fn),'tp':int(tp)}

def select_threshold(y,p):
    rows=[]
    for t in np.arange(.01,1,.01): rows.append(metrics(y,p,float(t)))
    return max(rows,key=lambda r:(r['f1'],r['recall']))


In [ ]:
# PyTorch runtime, loaders, weighted BCE, focal loss, training, and evaluation
subprocess.run([sys.executable,'-m','pip','install','-q','torch'],check=True)
import torch
from torch import nn
from torch.utils.data import TensorDataset,DataLoader

torch.manual_seed(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE=1024 if DEVICE.type=='cuda' else 512
train_loader=DataLoader(TensorDataset(torch.tensor(X_train),torch.tensor(y_train,dtype=torch.float32)),batch_size=BATCH_SIZE,shuffle=True)
val_tensor=torch.tensor(X_val,dtype=torch.float32,device=DEVICE)
test_tensor=torch.tensor(X_test,dtype=torch.float32,device=DEVICE)
pos_weight=torch.tensor([(y_train==0).sum()/(y_train==1).sum()],device=DEVICE,dtype=torch.float32)

class FocalLoss(nn.Module):
    def __init__(self,alpha=.75,gamma=2.): super().__init__(); self.alpha=alpha; self.gamma=gamma
    def forward(self,logits,target):
        bce=nn.functional.binary_cross_entropy_with_logits(logits,target,reduction='none')
        p=torch.sigmoid(logits); pt=torch.where(target==1,p,1-p); at=torch.where(target==1,self.alpha,1-self.alpha)
        return (at*(1-pt).pow(self.gamma)*bce).mean()

def predict(model,tensor):
    model.eval()
    with torch.no_grad(): return torch.sigmoid(model(tensor)).cpu().numpy()

def train_model(model,loss_fn,epochs=25,patience=5,lr=1e-3):
    model=model.to(DEVICE); opt=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=1e-4)
    best=-1.; best_state=None; stale=0; history=[]
    for epoch in range(1,epochs+1):
        model.train(); total=0.
        for xb,yb in train_loader:
            xb=xb.to(DEVICE); yb=yb.to(DEVICE); opt.zero_grad(set_to_none=True)
            logits=model(xb); loss=loss_fn(logits,yb); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),5.); opt.step(); total+=loss.item()*len(xb)
        vp=predict(model,val_tensor); score=average_precision_score(y_val,vp); history.append({'epoch':epoch,'loss':total/len(train_loader.dataset),'val_pr_auc':score})
        if score>best: best=score; best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; stale=0
        else: stale+=1
        if stale>=patience: break
    model.load_state_dict(best_state); return model,pd.DataFrame(history)
print('Device:',DEVICE,'Batch:',BATCH_SIZE)


## Research interpretation

This notebook does not assume a 13% fraud improvement. The published claim comes from other tabular benchmarks. Here, TabularGRPO is independently evaluated on ULB fraud decisioning.

The architecture contains numeric feature tokens, Transformer encoding, top-2 routed experts, a four-action policy, grouped action sampling, clipped policy updates, entropy regularization, KL regularization, and load balancing.


In [ ]:
GROUP_SIZE=8; ACTIONS=4; REWARD=torch.tensor([[1,-2,-3,-10],[-25,5,7,12]],dtype=torch.float32,device=DEVICE)
class Tokenizer(nn.Module):
    def __init__(self,n=30,d=48): super().__init__(); self.w=nn.Parameter(torch.randn(n,d)*.02); self.b=nn.Parameter(torch.zeros(n,d)); self.ids=nn.Parameter(torch.randn(n,d)*.02)
    def forward(self,x): return x.unsqueeze(-1)*self.w+self.b+self.ids
class Experts(nn.Module):
    def __init__(self,d=48,e=4): super().__init__(); self.router=nn.Linear(d,e); self.experts=nn.ModuleList([nn.Sequential(nn.Linear(d,96),nn.GELU(),nn.Linear(96,d)) for _ in range(e)]); self.last=None
    def forward(self,x):
        p=torch.softmax(self.router(x),-1); self.last=p; values,ids=p.topk(2,-1); out=torch.zeros_like(x)
        for k in range(2):
            for i,expert in enumerate(self.experts):
                m=ids[:,k]==i
                if m.any(): out[m]+=values[m,k:k+1]*expert(x[m])
        return out
    def balance(self): return ((self.last.mean(0)-1/self.last.shape[1])**2).mean()
class TabularGRPO(nn.Module):
    def __init__(self):
        super().__init__(); self.tok=Tokenizer(); self.cls=nn.Parameter(torch.zeros(1,1,48)); layer=nn.TransformerEncoderLayer(48,4,96,.1,batch_first=True,norm_first=True); self.encoder=nn.TransformerEncoder(layer,2); self.moe=Experts(); self.head=nn.Linear(48,ACTIONS)
    def forward(self,x):
        z=self.tok(x); z=self.encoder(torch.cat([self.cls.expand(len(x),-1,-1),z],1))[:,0]; z=z+self.moe(z); return self.head(z)
policy=TabularGRPO().to(DEVICE); reference=TabularGRPO().to(DEVICE); reference.load_state_dict(policy.state_dict()); opt=torch.optim.AdamW(policy.parameters(),2e-4); history=[]
for epoch in range(12):
    rewards_epoch=[]; zero=0; seen=0
    for xb,yb in train_loader:
        xb,yb=xb.to(DEVICE),yb.long().to(DEVICE)
        with torch.no_grad(): old_probs=torch.softmax(policy(xb),-1); actions=torch.distributions.Categorical(old_probs).sample((GROUP_SIZE,)).T; old_log=torch.log(old_probs.gather(1,actions)+1e-9); rewards=REWARD[yb.unsqueeze(1),actions]; std=rewards.std(1,keepdim=True); adv=(rewards-rewards.mean(1,keepdim=True))/(std+1e-6); zero+=(std<1e-6).sum().item(); seen+=len(xb)
        probs=torch.softmax(policy(xb),-1); log=torch.log(probs.gather(1,actions)+1e-9); ratio=torch.exp(log-old_log); clip=torch.minimum(ratio*adv,torch.clamp(ratio,.8,1.2)*adv).mean(); entropy=-(probs*torch.log(probs+1e-9)).sum(1).mean()
        with torch.no_grad(): ref=torch.softmax(reference(xb),-1)
        kl=(probs*(torch.log(probs+1e-9)-torch.log(ref+1e-9))).sum(1).mean(); loss=-clip+.02*kl-.01*entropy+.1*policy.moe.balance(); opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(policy.parameters(),5.); opt.step(); rewards_epoch.append(rewards.mean().item())
    history.append({'epoch':epoch+1,'mean_group_reward':float(np.mean(rewards_epoch)),'zero_variance_fraction':zero/seen,'kl':float(kl.item()),'entropy':float(entropy.item())})
policy.eval()
with torch.no_grad(): logits=policy(test_tensor); actions=logits.argmax(-1).cpu().numpy(); routing=policy.moe.last.cpu().numpy()
reward_values=np.array([[1,-2,-3,-10],[-25,5,7,12]])[y_test,actions]
final={'mean_reward':float(reward_values.mean()),'fraud_approval_rate':float((actions[y_test==1]==0).mean()),'false_decline_rate':float((actions[y_test==0]==3).mean()),'action_distribution':np.bincount(actions,minlength=4).tolist(),'expert_utilization':routing.mean(0).tolist(),'routing_entropy':float((-routing*np.log(routing+1e-9)).sum(1).mean())}
print(final)
RUN_DIR=RUNS_ROOT/(datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'_phase10_tabulargrpo'); RUN_DIR.mkdir(parents=True); torch.save(policy.state_dict(),RUN_DIR/'tabulargrpo.pt'); pd.DataFrame(history).to_csv(RUN_DIR/'history.csv',index=False); (RUN_DIR/'metrics.json').write_text(json.dumps(final,indent=2)); (RUN_DIR/'reward_table.json').write_text(json.dumps({0:[1,-2,-3,-10],1:[-25,5,7,12]},indent=2))


## Required conclusion

TabularGRPO is accepted only if it improves repeatable operational metrics without unacceptable false declines, review usage, reward instability, or expert collapse. A single run is exploratory evidence, not production proof.
